# RVC Voice Server — automated multi-speaker training + inference
**Thesis:** Integrating Speech-to-Text and RVC-Based Voice Conversion into an AI Assistant
**Author:** Nguyen Hoang Ngoc Bao — 24MSE23204

This notebook replaces the old two-notebook workflow (`train_rvc.ipynb` + `serve_rvc.ipynb`, still kept for
reference) with a single always-on HTTP server that the app calls directly. Instead of manually editing a
`SPEAKER_NAME` cell and running the whole pipeline by hand for every user, the app now uploads samples and
triggers training automatically — this notebook just needs to be **running**.

**API contract** (matches `voice/rvc_client.py` in the app):
- `GET  /health` → `{"status": "ok"}`
- `GET  /models` → `{"speakers": [...]}` (speaker_ids with a trained model available)
- `POST /train` — multipart: `speaker_id` + one or more `files` (wav/mp3 samples) → enqueues a training job,
  returns immediately: `{"status": "queued", "message": "..."}`
- `GET  /train_status/<speaker_id>` → `{"status": "queued"|"running"|"done"|"failed", "message": "..."}`
- `POST /convert` — multipart: `audio` (mp3/wav) + `speaker_id` + `pitch` + `index_rate` → converted WAV bytes
- `DELETE /models/<speaker_id>` → deletes that speaker's `.pth`/`.index` from Drive and evicts it from the in-memory cache (called when a user deletes their voice profile in the app)
- `GET /models/<speaker_id>/download` → zipped `.pth`+`.index` for that speaker, so the app can keep its own local backup copy under `voice_storage/` (called right after training finishes)
- `POST /baseline/f5tts` — multipart: `gen_text` (required), `ref_audio` (wav, optional — defaults to the bundled reference clip), `ref_text`, `speed` → WAV bytes.
  Zero-shot baseline (hynt/F5-TTS-Vietnamese-ViVoice, F5-TTS) used for the RQ2 MOS comparison against the trained RVC voice, and optionally as the base-voice engine instead of edge-TTS (select it via a profile's `base_tts_voice = "f5tts:default"`).
- `POST /transcribe` — multipart: `audio` (any format ffmpeg/librosa can decode) + `language` (optional, e.g. `"vi"`) → `{"text": str, "language": str}`.
  Speech-to-Text via `vinai/PhoWhisper-large` (VinAI's Vietnamese fine-tune of Whisper, thesis Section 2.1) — the **input** half of the voice loop, everything else above is the **output** half. The app falls back to a local, CPU-only `openai-whisper` model (`voice/stt.py`) whenever this endpoint is unset/unreachable.

Only one T4 GPU is available, so training jobs run one at a time through an internal queue — submitting a
second speaker while one is training just waits its turn instead of failing.

**Before running:**
1. `Runtime > Change runtime type > GPU (T4)`
2. Run all cells top-to-bottom
3. Copy the tunnel URL printed at the end into the app's admin page: `/admin/voice_models` → "Kết nối máy chủ RVC (Colab)"
4. Keep this notebook running — that's the one manual step. Everything after (sample upload → train → status →
   ready-to-speak) is automatic from the app side.


In [ ]:
# ── A. CONFIGURATION ─────────────────────────────────────────────────────────
SAMPLE_RATE   = 40000          # RVC v2 standard: 40 kHz
TOTAL_EPOCHS  = 200            # thesis target (Section 4.1 Stage 2)
BATCH_SIZE    = 8              # reduce to 4 if CUDA OOM on T4
SAVE_EVERY    = 20
F0_METHOD     = "rmvpe"        # RMVPE — most accurate per thesis Section 2.3
RVC_VERSION   = "v2"
SERVER_PORT   = 7860
PITCH         = 0              # semitone shift
INDEX_RATE    = 0.75           # FAISS blend (0.0 = ignore index, 1.0 = full retrieval)
PROTECT       = 0.33           # protect voiceless consonants from over-conversion

DRIVE_ROOT = "/content/drive/MyDrive/rvc_training"
MODELS_DIR = f"{DRIVE_ROOT}/models"

print("Configuration loaded — models are now trained/served dynamically per speaker_id")
print(f"  sample_rate={SAMPLE_RATE}  epochs={TOTAL_EPOCHS}  batch_size={BATCH_SIZE}  f0_method={F0_METHOD}")


In [ ]:
# ── B. MOUNT GOOGLE DRIVE ────────────────────────────────────────────────────
from google.colab import drive
import os

drive.mount("/content/drive")
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Drive mounted. Trained models are stored under: {MODELS_DIR}")


In [ ]:
# ── C. CLONE RVC + INSTALL DEPENDENCIES ──────────────────────────────────────
import os

if not os.path.exists("/content/RVC"):
    !git clone --depth=1 \
        https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI \
        /content/RVC 2>&1 | tail -5
else:
    print("RVC already cloned.")

%cd /content/RVC
!pip install -q -r requirements.txt
!pip install -q pydub librosa soundfile rvc-python flask

print("\n✓ Dependencies installed.")


In [ ]:
# ── D. GPU CHECK ─────────────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU (T4) and reconnect."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU  : {gpu_name}")
print(f"  VRAM : {vram_gb:.1f} GB")
if vram_gb < 10:
    print("⚠️ Low VRAM — consider reducing BATCH_SIZE to 4.")


In [ ]:
# ── E. DOWNLOAD PRETRAINED V2 MODELS ─────────────────────────────────────────
import os, urllib.request

HF_BASE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main"
ASSETS = {
    "assets/pretrained_v2/f0G40k.pth": f"{HF_BASE}/pretrained_v2/f0G40k.pth",
    "assets/pretrained_v2/f0D40k.pth": f"{HF_BASE}/pretrained_v2/f0D40k.pth",
    "assets/hubert/hubert_base.pt":     f"{HF_BASE}/hubert_base.pt",
    "assets/rmvpe/rmvpe.pt":            f"{HF_BASE}/rmvpe.pt",
}

for dest, url in ASSETS.items():
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest):
        print(f"  ✓ {os.path.basename(dest)} (cached)")
    else:
        print(f"  Downloading {os.path.basename(dest)} …")
        urllib.request.urlretrieve(url, dest)
        print(f"  ✓ {os.path.basename(dest)}  ({os.path.getsize(dest)/1e6:.0f} MB)")

print("\n✓ All pretrained assets ready.")


## Training pipeline (parameterized by `speaker_id`)
Same steps as the original `train_rvc.ipynb` (slice/normalize → preprocess → F0 → HuBERT features → train →
FAISS index → export), just wrapped in a function instead of hardcoded to one `SPEAKER_NAME` global, so the
worker thread below can run it for any speaker_id submitted by the app.

In [ ]:
# ── F. TRAINING PIPELINE FUNCTION ────────────────────────────────────────────
import glob, os, subprocess, sys, shutil
import numpy as np
import faiss
from pydub import AudioSegment
from pydub.silence import split_on_silence


def _slice_and_normalize(src: str, out_dir: str, sr: int = SAMPLE_RATE,
                          min_ms: int = 3000, max_ms: int = 8000) -> int:
    audio = AudioSegment.from_file(src)
    audio = audio.set_frame_rate(sr).set_channels(1)
    audio = audio.apply_gain(-20.0 - audio.dBFS)

    chunks = split_on_silence(
        audio, min_silence_len=300, silence_thresh=audio.dBFS - 16, keep_silence=150,
    )

    stem = os.path.splitext(os.path.basename(src))[0]
    saved, buf = 0, AudioSegment.empty()
    for chunk in chunks:
        buf += chunk
        while len(buf) >= min_ms:
            seg = buf[:max_ms]
            buf = buf[max_ms:]
            seg.export(os.path.join(out_dir, f"{stem}_{saved:04d}.wav"), format="wav")
            saved += 1
    if len(buf) >= min_ms:
        buf.export(os.path.join(out_dir, f"{stem}_{saved:04d}.wav"), format="wav")
        saved += 1
    return saved


def _run_step(cmd, label):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{label} failed:\n{result.stderr[-1500:]}")
    return result.stdout


def train_speaker(speaker_id: str, raw_dir: str, progress_cb=None):
    """
    Full pipeline for one speaker: slice/normalize -> preprocess -> F0 ->
    HuBERT features -> train RVC v2 -> FAISS index -> export to Drive.
    raw_dir must already contain the uploaded raw wav/mp3 samples.
    """
    def report(msg):
        print(f"[{speaker_id}] {msg}")
        if progress_cb:
            progress_cb(msg)

    os.chdir("/content/RVC")
    sliced_dir = f"/content/dataset_{speaker_id}_sliced"
    os.makedirs(sliced_dir, exist_ok=True)
    exp_dir = f"logs/{speaker_id}"
    os.makedirs(exp_dir, exist_ok=True)

    report("Slicing & normalizing samples…")
    sources = (glob.glob(os.path.join(raw_dir, "*.wav")) +
               glob.glob(os.path.join(raw_dir, "*.mp3")) +
               glob.glob(os.path.join(raw_dir, "*.webm")) +
               glob.glob(os.path.join(raw_dir, "*.ogg")) +
               glob.glob(os.path.join(raw_dir, "*.m4a")))
    if not sources:
        raise RuntimeError(f"No audio files found in {raw_dir}")
    total_segs = sum(_slice_and_normalize(f, sliced_dir) for f in sources)
    report(f"{total_segs} segments produced.")
    if total_segs < 20:
        report("⚠️ Few segments — quality may be lower than ideal, continuing anyway.")

    report("Preprocessing…")
    _run_step([sys.executable, "trainset_preprocess_pipeline_print.py",
               sliced_dir, str(SAMPLE_RATE), "4", exp_dir, "False", "3.7"], "Preprocessing")

    report("Extracting F0 (RMVPE)…")
    _run_step([sys.executable, "extract_f0_print.py", exp_dir, "4", F0_METHOD], "F0 extraction")

    report("Extracting HuBERT features…")
    _run_step([sys.executable, "extract_feature_print.py",
               "cuda:0", "1", "0", "0", exp_dir, RVC_VERSION], "Feature extraction")

    report("Building training filelist…")
    gt_wavs_dir = os.path.join(exp_dir, "0_gt_wavs")
    feature_dir = os.path.join(exp_dir, "3_feature768")
    f0_dir      = os.path.join(exp_dir, "2a_f0")
    f0nsf_dir   = os.path.join(exp_dir, "2b-f0nsf")

    wav_map = {os.path.splitext(os.path.basename(w))[0]: w
               for w in sorted(glob.glob(os.path.join(gt_wavs_dir, "*.wav")))}
    feat_map = {os.path.splitext(os.path.basename(f))[0]: f
                for f in sorted(glob.glob(os.path.join(feature_dir, "*.npy")))}
    common = sorted(set(wav_map) & set(feat_map))
    if not common:
        raise RuntimeError("No matching wav/feature pairs — preprocessing or feature extraction failed.")

    lines = [f"{wav_map[s]}|{feat_map[s]}|{os.path.join(f0_dir, s + '.wav.npy')}|"
             f"{os.path.join(f0nsf_dir, s + '.wav.npy')}|0" for s in common]
    filelist_path = os.path.join(exp_dir, "filelist.txt")
    with open(filelist_path, "w") as fh:
        fh.write("\n".join(lines))
    report(f"Filelist: {len(lines)} entries.")

    report(f"Training {TOTAL_EPOCHS} epochs (this takes ~30-60 min on a T4)…")
    result = subprocess.run([
        sys.executable, "train.py",
        "-e", speaker_id, "-sr", "40k", "-f0", "1", "-bs", str(BATCH_SIZE),
        "-g", "0", "-te", str(TOTAL_EPOCHS), "-se", str(SAVE_EVERY),
        "-pg", "assets/pretrained_v2/f0G40k.pth", "-pd", "assets/pretrained_v2/f0D40k.pth",
        "-l", "1", "-c", "0", "-sw", "0", "-v", RVC_VERSION,
    ])
    if result.returncode != 0:
        raise RuntimeError("Training failed — check Colab logs above.")
    report("Training complete.")

    report("Building FAISS index…")
    feat_files = sorted(glob.glob(os.path.join(feature_dir, "*.npy")))
    features = np.concatenate([np.load(f) for f in feat_files], axis=0).astype("float32")
    n_ivf = max(min(int(16 * np.sqrt(len(features))), len(features) // 39 + 1), 4)
    index = faiss.index_factory(features.shape[1], f"IVF{n_ivf},Flat")
    index.train(features)
    index.add(features)
    index_path = os.path.join(exp_dir, f"{speaker_id}.index")
    faiss.write_index(index, index_path)

    def find_latest(pattern):
        files = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)
        return files[0] if files else None

    pth_path = find_latest(f"{exp_dir}/{speaker_id}_*.pth") or find_latest(f"{exp_dir}/G_*.pth")
    if not pth_path:
        raise RuntimeError("No .pth checkpoint found after training.")

    out_pth   = os.path.join(MODELS_DIR, f"{speaker_id}.pth")
    out_index = os.path.join(MODELS_DIR, f"{speaker_id}.index")
    shutil.copy2(pth_path, out_pth)
    shutil.copy2(index_path, out_index)
    report(f"Exported model → {out_pth}")

    shutil.rmtree(sliced_dir, ignore_errors=True)
    return out_pth, out_index


print("✓ train_speaker() ready.")


In [ ]:
# ── G. MODEL CACHE + CONVERT (multi-speaker, loaded on demand) ───────────────
from rvc_python.infer import RVCInference

_model_cache = {}

def get_model(speaker_id: str):
    if speaker_id in _model_cache:
        return _model_cache[speaker_id]
    pth_path   = os.path.join(MODELS_DIR, f"{speaker_id}.pth")
    index_path = os.path.join(MODELS_DIR, f"{speaker_id}.index")
    if not (os.path.exists(pth_path) and os.path.exists(index_path)):
        return None
    rvc = RVCInference(device="cuda:0")
    rvc.load_model(pth_path, index_path)
    _model_cache[speaker_id] = rvc
    return rvc


def list_trained_speakers():
    return sorted({
        os.path.splitext(os.path.basename(p))[0]
        for p in glob.glob(os.path.join(MODELS_DIR, "*.pth"))
    })


print("✓ Model cache ready. Trained speakers so far:", list_trained_speakers())


In [ ]:
# ── H. TRAINING JOB QUEUE (one GPU → one job at a time) ──────────────────────
import queue, threading, shutil

TRAIN_JOBS = {}   # speaker_id -> {"status": "queued"|"running"|"done"|"failed", "message": str}
_train_queue = queue.Queue()
_jobs_lock = threading.Lock()


def _set_job(speaker_id, **kwargs):
    with _jobs_lock:
        TRAIN_JOBS.setdefault(speaker_id, {}).update(kwargs)


def _worker_loop():
    while True:
        speaker_id, raw_dir = _train_queue.get()
        _set_job(speaker_id, status="running", message="Bat dau huan luyen...")
        try:
            train_speaker(speaker_id, raw_dir, progress_cb=lambda m: _set_job(speaker_id, message=m))
            _set_job(speaker_id, status="done", message="Huan luyen hoan tat.")
        except Exception as exc:
            print(f"[{speaker_id}] TRAINING FAILED: {exc}")
            _set_job(speaker_id, status="failed", message=str(exc))
        finally:
            shutil.rmtree(raw_dir, ignore_errors=True)
            _train_queue.task_done()


threading.Thread(target=_worker_loop, daemon=True).start()
print("✓ Training worker thread started (processes one speaker at a time).")


## Baseline: F5-TTS-Vietnamese-ViVoice (zero-shot voice cloning)
Adds `hynt/F5-TTS-Vietnamese-ViVoice` — a Vietnamese fine-tune of F5-TTS (flow-matching,
trained on ~1000h of Vietnamese speech, ViVoice dataset) — as a second **zero-shot**
baseline next to XTTS-v2 for the thesis's RQ2 comparison (does per-speaker RVC training
beat zero-shot cloning on naturalness / speaker similarity?). Unlike RVC, it needs no
per-speaker training: it clones a voice from a single ~6-10s reference clip + its exact
transcript, supplied at inference time.

It is also wired in as an optional **base-voice engine** in the app (`voice/tts.py`),
selectable in place of edge-TTS by setting a profile's `base_tts_voice` to `"f5tts:default"`
— useful for demoing whether a more natural Vietnamese base signal improves the final
RVC-converted output.

**Before this section is usable:** upload one clean reference clip to
`f5tts_assets/reference.wav` under `DRIVE_ROOT` and set `REF_TEXT_F5TTS` (next cell) to
its exact transcript — F5-TTS has no default voice of its own, it always clones from a
reference.

In [ ]:
# ── L1. INSTALL F5-TTS-VIETNAMESE-VIVOICE ────────────────────────────────────
import os

F5TTS_DIR = "/content/F5-TTS-Vietnamese"
if not os.path.exists(F5TTS_DIR):
    !git clone --depth=1 https://github.com/nguyenthienhy/F5-TTS-Vietnamese {F5TTS_DIR} 2>&1 | tail -5
else:
    print("F5-TTS-Vietnamese already cloned.")

%cd {F5TTS_DIR}
!pip install -q -e .
%cd /content/RVC

print("\n\u2713 F5-TTS-Vietnamese installed.")


In [ ]:
# ── L2. DOWNLOAD hynt/F5-TTS-Vietnamese-ViVoice CHECKPOINT ───────────────────
from huggingface_hub import hf_hub_download, list_repo_files

F5TTS_REPO        = "hynt/F5-TTS-Vietnamese-ViVoice"
F5TTS_ASSETS_DIR  = f"{DRIVE_ROOT}/f5tts_assets"
os.makedirs(F5TTS_ASSETS_DIR, exist_ok=True)

_repo_files = list_repo_files(F5TTS_REPO)
print("Files in repo:", _repo_files)

def _pick(suffixes):
    for suf in suffixes:
        for fname in _repo_files:
            if fname.endswith(suf):
                return fname
    return None

_ckpt_name  = _pick([".pt", ".safetensors"]) or "model_last.pt"
_vocab_name = _pick(["vocab.txt"])

F5TTS_CKPT = hf_hub_download(F5TTS_REPO, _ckpt_name, local_dir=F5TTS_ASSETS_DIR)

if _vocab_name:
    F5TTS_VOCAB = hf_hub_download(F5TTS_REPO, _vocab_name, local_dir=F5TTS_ASSETS_DIR)
else:
    # Some releases of this checkpoint ship the vocab under config.json instead of a
    # plain vocab.txt (see the model card) -- download it and rename locally.
    cfg_path = hf_hub_download(F5TTS_REPO, "config.json", local_dir=F5TTS_ASSETS_DIR)
    F5TTS_VOCAB = os.path.join(F5TTS_ASSETS_DIR, "vocab.txt")
    os.replace(cfg_path, F5TTS_VOCAB)

print(f"\u2713 Checkpoint: {F5TTS_CKPT}")
print(f"\u2713 Vocab     : {F5TTS_VOCAB}")


In [ ]:
# ── L3. LOAD MODEL + synthesize_f5tts() ───────────────────────────────────────
import io
import soundfile as sf
from f5_tts.api import F5TTS

f5tts_model = F5TTS(model_type="F5TTS_Base", ckpt_file=F5TTS_CKPT, vocab_file=F5TTS_VOCAB, device="cuda")

# Reference clip for zero-shot cloning -- ~6-10s clean, single-speaker Vietnamese audio
# plus its exact transcript. Upload the wav to F5TTS_ASSETS_DIR/reference.wav and fill
# in REF_TEXT_F5TTS below (a training sample from one of your RVC speakers works fine).
REF_AUDIO_F5TTS = f"{F5TTS_ASSETS_DIR}/reference.wav"
REF_TEXT_F5TTS  = ""  # <-- set this to the exact transcript of REF_AUDIO_F5TTS


def synthesize_f5tts(gen_text: str, ref_audio_path: str = None, ref_text: str = None,
                      speed: float = 1.0) -> bytes:
    """Zero-shot Vietnamese TTS via F5-TTS-Vietnamese-ViVoice. Returns WAV bytes."""
    ref_audio_path = ref_audio_path or REF_AUDIO_F5TTS
    ref_text = ref_text if ref_text is not None else REF_TEXT_F5TTS
    if not os.path.exists(ref_audio_path):
        raise FileNotFoundError(f"F5-TTS reference clip not found: {ref_audio_path}")
    if not ref_text:
        raise ValueError("REF_TEXT_F5TTS is empty -- set it to the reference clip's exact transcript.")

    wav, sr, _ = f5tts_model.infer(ref_file=ref_audio_path, ref_text=ref_text,
                                    gen_text=gen_text, speed=speed)
    buf = io.BytesIO()
    sf.write(buf, wav, sr, format="WAV")
    return buf.getvalue()


print("\u2713 synthesize_f5tts() ready.",
      "Set REF_TEXT_F5TTS and upload reference.wav before calling it." if not REF_TEXT_F5TTS else "")


# ── I. FLASK SERVER — implements the API contract expected by voice/rvc_client.py
import io, os, tempfile, time, uuid, zipfile
from flask import Flask, request, jsonify, send_file

server = Flask(__name__)


@server.get("/health")
def health():
    return jsonify({"status": "ok"})


@server.get("/models")
def models_route():
    return jsonify({"speakers": list_trained_speakers()})


@server.get("/models/<speaker_id>/download")
def download_model_route(speaker_id):
    """Zips the trained .pth+.index for this speaker so the app can back it up locally."""
    pth_path   = os.path.join(MODELS_DIR, f"{speaker_id}.pth")
    index_path = os.path.join(MODELS_DIR, f"{speaker_id}.index")
    if not (os.path.exists(pth_path) and os.path.exists(index_path)):
        return jsonify({"error": f"Khong tim thay model da huan luyen cho {speaker_id}"}), 404

    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pth_path, arcname=f"{speaker_id}.pth")
        zf.write(index_path, arcname=f"{speaker_id}.index")
    buf.seek(0)
    return send_file(buf, mimetype="application/zip", as_attachment=True,
                      download_name=f"{speaker_id}.zip")


@server.delete("/models/<speaker_id>")
def delete_model_route(speaker_id):
    """Deletes a trained speaker's .pth/.index from Drive and the in-memory cache."""
    _model_cache.pop(speaker_id, None)
    pth_path   = os.path.join(MODELS_DIR, f"{speaker_id}.pth")
    index_path = os.path.join(MODELS_DIR, f"{speaker_id}.index")
    removed = False
    for p in (pth_path, index_path):
        if os.path.exists(p):
            os.remove(p)
            removed = True
    with _jobs_lock:
        TRAIN_JOBS.pop(speaker_id, None)
    return jsonify({"status": "ok", "removed": removed})


@server.post("/transcribe")
def transcribe_route():
    """
    Speech-to-Text via PhoWhisper (Vietnamese-tuned) — input half of the voice loop
    (see /baseline/f5tts and /convert below for the output half).
    multipart: audio (any format ffmpeg/librosa can decode), language (optional, e.g. "vi").
    """
    if "audio" not in request.files:
        return jsonify({"error": "thieu audio"}), 400
    if asr_pipeline is None:
        return jsonify({"error": "Mo hinh ASR chua san sang"}), 503

    language   = request.form.get("language") or None
    audio_file = request.files["audio"]
    suffix     = os.path.splitext(audio_file.filename or "")[1] or ".webm"

    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
        audio_file.save(f.name)
        tmp_path = f.name

    try:
        kwargs = {"ignore_warning": True}
        if language:
            kwargs["generate_kwargs"] = {"language": language}
        result = asr_pipeline(tmp_path, **kwargs)
        return jsonify({
            "text": result["text"].strip(),
            "language": language or "vi",
            "engine": f"phowhisper:{ASR_MODEL_NAME.rsplit('/', 1)[-1]}",
        })
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass


@server.post("/baseline/f5tts")
def baseline_f5tts_route():
    """
    Zero-shot Vietnamese TTS baseline (thesis RQ2: TTS+RVC vs zero-shot cloning).
    multipart: gen_text (required), ref_audio (wav, optional -- defaults to
    REF_AUDIO_F5TTS), ref_text (optional -- defaults to REF_TEXT_F5TTS), speed.
    """
    gen_text = request.form.get("gen_text")
    if not gen_text:
        return jsonify({"error": "thieu gen_text"}), 400

    speed = float(request.form.get("speed", 1.0))
    ref_text = request.form.get("ref_text") or None

    ref_audio_path = REF_AUDIO_F5TTS
    tmp_ref = None
    if "ref_audio" in request.files:
        tmp_ref = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        request.files["ref_audio"].save(tmp_ref.name)
        ref_audio_path = tmp_ref.name

    try:
        wav_bytes = synthesize_f5tts(gen_text, ref_audio_path=ref_audio_path,
                                      ref_text=ref_text, speed=speed)
        return send_file(io.BytesIO(wav_bytes), mimetype="audio/wav",
                          as_attachment=False, download_name="f5tts.wav")
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500
    finally:
        if tmp_ref:
            try:
                os.unlink(tmp_ref.name)
            except OSError:
                pass


@server.post("/train")
def train_route():
    speaker_id = request.form.get("speaker_id")
    files = request.files.getlist("files")
    if not speaker_id or not files:
        return jsonify({"status": "error", "message": "Thieu speaker_id hoac file mau."}), 400

    with _jobs_lock:
        existing_status = TRAIN_JOBS.get(speaker_id, {}).get("status")
    if existing_status in ("queued", "running"):
        return jsonify({"status": "error", "message": "Giong noi nay dang duoc xu ly."}), 400

    raw_dir = tempfile.mkdtemp(prefix=f"raw_{speaker_id}_")
    for f in files:
        f.save(os.path.join(raw_dir, f.filename or f"{uuid.uuid4()}.wav"))

    _set_job(speaker_id, status="queued", message="Dang cho trong hang doi huan luyen.")
    _train_queue.put((speaker_id, raw_dir))
    return jsonify({"status": "queued", "message": "Da them vao hang doi huan luyen."})


@server.get("/train_status/<speaker_id>")
def train_status_route(speaker_id):
    with _jobs_lock:
        job = TRAIN_JOBS.get(speaker_id)
    if not job:
        return jsonify({"status": "unknown"})
    return jsonify(job)


@server.post("/convert")
def convert_route():
    speaker_id = request.form.get("speaker_id")
    if "audio" not in request.files or not speaker_id:
        return jsonify({"error": "thieu audio hoac speaker_id"}), 400

    rvc = get_model(speaker_id)
    if rvc is None:
        return jsonify({"error": f"Chua co model da huan luyen cho {speaker_id}"}), 404

    pitch      = int(request.form.get("pitch", PITCH))
    index_rate = float(request.form.get("index_rate", INDEX_RATE))

    audio_bytes = request.files["audio"].read()
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as fin:
        fin.write(audio_bytes)
        in_path = fin.name
    out_path = in_path.replace(".mp3", "_rvc.wav")

    try:
        rvc.infer_file(input_path=in_path, output_path=out_path, f0_up_key=pitch,
                        f0_method=F0_METHOD, index_rate=index_rate, protect=PROTECT)
        return send_file(out_path, mimetype="audio/wav", as_attachment=False)
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500
    finally:
        try:
            os.unlink(in_path)
        except OSError:
            pass


def _run_server():
    server.run(host="0.0.0.0", port=SERVER_PORT, debug=False)


threading.Thread(target=_run_server, daemon=True).start()
time.sleep(1.5)
print(f"✓ Voice server listening on port {SERVER_PORT}")
print("  Routes: /health  /models  /transcribe  /train  /train_status/<id>  /convert  /baseline/f5tts")


In [ ]:
# ── M. LOAD PHOWHISPER (VIETNAMESE ASR) ──────────────────────────────────────
import gc
import torch
from transformers import pipeline

ASR_MODEL_NAME = "vinai/PhoWhisper-large"   # swap for -small / -medium to trade accuracy for latency
ASR_CHUNK_S    = 30

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

print(f"Loading ASR model: {ASR_MODEL_NAME} ...")
try:
    asr_pipeline = pipeline(
        "automatic-speech-recognition",
        model=ASR_MODEL_NAME,
        chunk_length_s=ASR_CHUNK_S,
        device="cuda" if torch.cuda.is_available() else "cpu",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        model_kwargs={"use_safetensors": False, "low_cpu_mem_usage": True},
    )
    print("✓ PhoWhisper ready.")
except Exception as e:
    asr_pipeline = None
    print(f"⚠️ Failed to load PhoWhisper: {e} — /transcribe will return 503 until this is fixed.")


In [ ]:
# ── I. FLASK SERVER — implements the API contract expected by voice/rvc_client.py
import io, os, tempfile, time, uuid, zipfile
from flask import Flask, request, jsonify, send_file

server = Flask(__name__)


@server.get("/health")
def health():
    return jsonify({"status": "ok"})


@server.get("/models")
def models_route():
    return jsonify({"speakers": list_trained_speakers()})


@server.get("/models/<speaker_id>/download")
def download_model_route(speaker_id):
    """Zips the trained .pth+.index for this speaker so the app can back it up locally."""
    pth_path   = os.path.join(MODELS_DIR, f"{speaker_id}.pth")
    index_path = os.path.join(MODELS_DIR, f"{speaker_id}.index")
    if not (os.path.exists(pth_path) and os.path.exists(index_path)):
        return jsonify({"error": f"Khong tim thay model da huan luyen cho {speaker_id}"}), 404

    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pth_path, arcname=f"{speaker_id}.pth")
        zf.write(index_path, arcname=f"{speaker_id}.index")
    buf.seek(0)
    return send_file(buf, mimetype="application/zip", as_attachment=True,
                      download_name=f"{speaker_id}.zip")


@server.delete("/models/<speaker_id>")
def delete_model_route(speaker_id):
    """Deletes a trained speaker's .pth/.index from Drive and the in-memory cache."""
    _model_cache.pop(speaker_id, None)
    pth_path   = os.path.join(MODELS_DIR, f"{speaker_id}.pth")
    index_path = os.path.join(MODELS_DIR, f"{speaker_id}.index")
    removed = False
    for p in (pth_path, index_path):
        if os.path.exists(p):
            os.remove(p)
            removed = True
    with _jobs_lock:
        TRAIN_JOBS.pop(speaker_id, None)
    return jsonify({"status": "ok", "removed": removed})


@server.post("/baseline/f5tts")
def baseline_f5tts_route():
    """
    Zero-shot Vietnamese TTS baseline (thesis RQ2: TTS+RVC vs zero-shot cloning).
    multipart: gen_text (required), ref_audio (wav, optional -- defaults to
    REF_AUDIO_F5TTS), ref_text (optional -- defaults to REF_TEXT_F5TTS), speed.
    """
    gen_text = request.form.get("gen_text")
    if not gen_text:
        return jsonify({"error": "thieu gen_text"}), 400

    speed = float(request.form.get("speed", 1.0))
    ref_text = request.form.get("ref_text") or None

    ref_audio_path = REF_AUDIO_F5TTS
    tmp_ref = None
    if "ref_audio" in request.files:
        tmp_ref = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        request.files["ref_audio"].save(tmp_ref.name)
        ref_audio_path = tmp_ref.name

    try:
        wav_bytes = synthesize_f5tts(gen_text, ref_audio_path=ref_audio_path,
                                      ref_text=ref_text, speed=speed)
        return send_file(io.BytesIO(wav_bytes), mimetype="audio/wav",
                          as_attachment=False, download_name="f5tts.wav")
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500
    finally:
        if tmp_ref:
            try:
                os.unlink(tmp_ref.name)
            except OSError:
                pass


@server.post("/train")
def train_route():
    speaker_id = request.form.get("speaker_id")
    files = request.files.getlist("files")
    if not speaker_id or not files:
        return jsonify({"status": "error", "message": "Thieu speaker_id hoac file mau."}), 400

    with _jobs_lock:
        existing_status = TRAIN_JOBS.get(speaker_id, {}).get("status")
    if existing_status in ("queued", "running"):
        return jsonify({"status": "error", "message": "Giong noi nay dang duoc xu ly."}), 400

    raw_dir = tempfile.mkdtemp(prefix=f"raw_{speaker_id}_")
    for f in files:
        f.save(os.path.join(raw_dir, f.filename or f"{uuid.uuid4()}.wav"))

    _set_job(speaker_id, status="queued", message="Dang cho trong hang doi huan luyen.")
    _train_queue.put((speaker_id, raw_dir))
    return jsonify({"status": "queued", "message": "Da them vao hang doi huan luyen."})


@server.get("/train_status/<speaker_id>")
def train_status_route(speaker_id):
    with _jobs_lock:
        job = TRAIN_JOBS.get(speaker_id)
    if not job:
        return jsonify({"status": "unknown"})
    return jsonify(job)


@server.post("/convert")
def convert_route():
    speaker_id = request.form.get("speaker_id")
    if "audio" not in request.files or not speaker_id:
        return jsonify({"error": "thieu audio hoac speaker_id"}), 400

    rvc = get_model(speaker_id)
    if rvc is None:
        return jsonify({"error": f"Chua co model da huan luyen cho {speaker_id}"}), 404

    pitch      = int(request.form.get("pitch", PITCH))
    index_rate = float(request.form.get("index_rate", INDEX_RATE))

    audio_bytes = request.files["audio"].read()
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as fin:
        fin.write(audio_bytes)
        in_path = fin.name
    out_path = in_path.replace(".mp3", "_rvc.wav")

    try:
        rvc.infer_file(input_path=in_path, output_path=out_path, f0_up_key=pitch,
                        f0_method=F0_METHOD, index_rate=index_rate, protect=PROTECT)
        return send_file(out_path, mimetype="audio/wav", as_attachment=False)
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500
    finally:
        try:
            os.unlink(in_path)
        except OSError:
            pass


def _run_server():
    server.run(host="0.0.0.0", port=SERVER_PORT, debug=False)


threading.Thread(target=_run_server, daemon=True).start()
time.sleep(1.5)
print(f"✓ Voice server listening on port {SERVER_PORT}")
print("  Routes: /health  /models  /train  /train_status/<id>  /convert")


In [ ]:
# ── J. CLOUDFLARED TUNNEL — exposes the server publicly over HTTPS ──────────
import os, re, subprocess, threading, time

if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared
    print("✓ cloudflared downloaded.")
else:
    print("✓ cloudflared already present.")

_tunnel_url = []


def _run_tunnel():
    proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", f"http://localhost:{SERVER_PORT}", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
        if m and not _tunnel_url:
            _tunnel_url.append(m.group(0))


threading.Thread(target=_run_tunnel, daemon=True).start()

print("Starting cloudflared tunnel… (waiting up to 30s)")
for _ in range(30):
    if _tunnel_url:
        break
    time.sleep(1)

if not _tunnel_url:
    raise RuntimeError("Tunnel URL not detected — check cloudflared output above.")

RVC_ENDPOINT = _tunnel_url[0]
print(f"\n✓ Tunnel active: {RVC_ENDPOINT}")


In [ ]:
# ── K. COPY THIS URL INTO THE APP ─────────────────────────────────────────────
# Admin page: /admin/voice_models -> "Ket noi may chu RVC (Colab)"
# DO NOT stop or restart the runtime -- cloudflared will assign a new URL and
# the app's stored endpoint will need to be updated again.

print("=" * 70)
print("  Paste this URL into the app's admin page:")
print("  /admin/voice_models  ->  'Ket noi may chu RVC (Colab)'")
print("=" * 70)
print()
print(f"  {RVC_ENDPOINT}")
print()
print("Keep this notebook running -- training jobs submitted from the app are")
print("processed automatically (one at a time) as long as this session stays")
print("alive. The endpoint goes offline as soon as the runtime stops.")

import urllib.request, json, time as _time
_time.sleep(2)
try:
    with urllib.request.urlopen(f"{RVC_ENDPOINT}/health", timeout=10) as r:
        print(f"\n✓ /health check passed: {json.loads(r.read())}")
except Exception as exc:
    print(f"\n⚠️ /health check failed: {exc}")
    print("   The server may still be starting -- retry in a few seconds.")
